In [0]:
dbutils.widgets.text("env", "test")

In [0]:
%run /Users/victoria18203@gmail.com/aw_project/ingestion/bronze_ingestion

In [0]:
%run /Users/victoria18203@gmail.com/aw_project/silver/silver_transformation

In [0]:
from pyspark.sql.functions import col, sum, when, to_date, hour, avg, count
from delta.tables import DeltaTable
from datetime import datetime

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS test.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS test.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS test.gold")

In [0]:
#clean up
spark.sql("DROP TABLE IF EXISTS test.bronze.yellow_taxi_raw")
spark.sql("DROP TABLE IF EXISTS test.silver.yellow_taxi_clean")
dbutils.fs.rm("abfss://landing@nyctaxiistorage.dfs.core.windows.net/test/", True)
print("Setup complete - clean environment ready")

In [0]:
#CREATE FAKE DATA
from datetime import datetime

# 1. Define the raw data as a list of tuples
data = [
    (1, datetime(2023, 1, 1, 9, 0), datetime(2023, 1, 1, 9, 15), 1, 2.5, 12.0, 1, 15.30, 1, 1, 1.0),
    (1, datetime(2023, 1, 1, 10, 30), datetime(2023, 1, 1, 10, 45), 2, 3.1, 14.5, 2, 16.80, 1, 2, 1.0),
    (2, datetime(2023, 1, 2, 14, 0), datetime(2023, 1, 2, 14, 25), 1, 5.2, 22.0, 1, 26.50, 2, 1, 1.0),
    (2, datetime(2023, 1, 2, 15, 10), datetime(2023, 1, 2, 15, 20), 4, 1.2, 8.5, 1, 10.20, 2, 3, 1.0),
    (1, datetime(2023, 1, 3, 0, 5), datetime(2023, 1, 3, 0, 15), 1, 2.0, 9.0, 2, 11.00, 1, 1, 1.0),
    (2, datetime(2023, 1, 3, 18, 30), datetime(2023, 1, 3, 19, 0), 1, 10.5, 45.0, 1, 52.00, 3, 2, 1.0),
    (1, datetime(2023, 1, 1, 9, 0), datetime(2023, 1, 1, 9, 15), 1, 2.5, 12.0, 1, 15.30, 1, 1, 1.0)
]

# 2. Define the column names in order
columns = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "fare_amount",
    "payment_type", "total_amount", "PULocationID", "DOLocationID", "RatecodeID"
]

# 3. Create the DataFrame
df = spark.createDataFrame(data, schema=columns)
# Display the results
df.show()
df.printSchema()

In [0]:
#WRITE TO LANDING
test_landing_path = "abfss://landing@nyctaxiistorage.dfs.core.windows.net/test/"

df.write.format("parquet").mode("overwrite").save(test_landing_path)
print("Test data written to landing")

In [0]:
spark.sql("DROP TABLE IF EXISTS test.bronze.yellow_taxi_raw")
spark.sql("DROP TABLE IF EXISTS test.silver.yellow_taxi_clean")

In [0]:
history_loader(test_landing_path, "yellow_taxi_raw", catalog="test")

In [0]:
count = spark.read.table("test.bronze.yellow_taxi_raw").count()
print(f"Bronze count after history_loader: {count}")

In [0]:
#RUN INGESTION ON TEST DATA
#row count
actual = spark.read.table("test.bronze.yellow_taxi_raw").count()
assert actual == 7, f"Test failed: expected 7 rows in bronze, got {actual}"

In [0]:
run_silver_transformation(catalog="test")

In [0]:
#RUN SILVER TRANSFORMATION ON TEST DATA
#silver row count
bronze_count = spark.read.table("test.bronze.yellow_taxi_raw").count()
silver_count = spark.read.table("test.silver.yellow_taxi_clean").count()
assert silver_count == 6, f"Test failed: expected 6 rows in silver, got {silver_count}"

In [0]:
#ASSERT RESULTS
#column existence
columns = spark.read.table("test.bronze.yellow_taxi_raw").columns
assert "ingestion_timestamp" in columns, "Test failed: ingestion_timestamp column missing"
assert "source_file" in columns, "Test failed: source_file column missing"